# Lab 3: 验证与安全层 — Safety & Permission

对应 Harness 六层架构：第 ❸ 层（教学顺序调整：先学安全，再学记忆）

## 学习目标

- 理解 **权限中间件模式** — 在工具执行前插入安全拦截层
- 实现 `allow` / `deny` / `ask` 三级权限控制
- 实现简单的 Bash 命令安全分类器
- 理解 **HITL（Human-in-the-Loop）** 设计哲学

## 痛点回顾

在 Lab 3 中，我们给 agent 装上了上下文与内存管理。但它仍然**没有任何安全检查**！

如果你对 agent 说"删除所有文件"，它会毫不犹豫地执行 `rm -rf *`。

**本 Lab 的目标：给 agent 装上"安全带"。**

## 对应 Claude Code 源码

- `utils/permissions/` — 权限规则引擎
- `hooks/useCanUseTool.js` — 工具使用权限检查 hook
- Bash Classifier — 命令安全性分类器

Claude Code 有 4 种权限模式：
- `default` — 危险操作需要用户确认
- `plan` — 只读模式，所有写操作需审批
- `auto` — 分类器自动判断是否安全
- `bypassPermissions` — 全部放行（仅开发调试用）


---
## 第一步：环境准备 + 回顾 Lab 2-3 代码

我们需要 Lab 2 的工具定义和 agent loop。以下 cell 快速重建 Lab 2 的核心代码。


In [ ]:
# === 环境准备 + Lab 2-3 代码回顾 ===
from anthropic import AnthropicBedrock
client = AnthropicBedrock(aws_region="us-west-2")
MODEL = "global.anthropic.claude-sonnet-4-6"

# --- 三个核心工具（来自 Lab 2)---
from utils.tools import WriteFileTool,ReadFileTool,BashTool,Tool

TOOLS = [ReadFileTool(), WriteFileTool(), BashTool()]
print(f"✓ Lab 2-3 代码回顾完成，{len(TOOLS)} 个工具就绪")


✓ Lab 2-3 代码回顾完成，3 个工具就绪


---
## 第二步：实现权限规则引擎

权限系统的核心思想：**在工具执行前插入一个拦截层（中间件）**。

每条权限规则包含：
- `tool_name` — 匹配哪个工具
- `behavior` — 三种行为：
  - `allow` — 直接放行（如只读操作）
  - `deny` — 直接拒绝（如禁止删除系统文件）
  - `ask` — 暂停，等待用户确认（HITL）

这对应 Claude Code 中 `utils/permissions/` 的权限规则结构。

In [3]:
# 权限规则引擎（对应 Claude Code 的 utils/permissions/）

from dataclasses import dataclass
from typing import Literal

@dataclass
class PermissionRule:
    """一条权限规则"""
    tool_name: str                                    # 匹配的工具名
    behavior: Literal["allow", "deny", "ask"]         # 行为
    pattern: str | None = None                        # 可选的匹配模式

class PermissionChecker:
    """权限检查器。对应 Claude Code 的权限中间件。"""
    
    def __init__(self):
        # 默认规则
        self.rules: list[PermissionRule] = [
            PermissionRule("read_file", "allow"),    # 只读操作：自动放行
            PermissionRule("write_file", "ask"),     # 写文件：需要确认
            PermissionRule("bash", "ask"),           # bash命令：需要确认（后面用分类器细化）
        ]
    
    def check(self, tool_name: str, tool_input: dict) -> str:
        """检查工具调用是否被允许。返回 'allow' / 'deny' / 'ask'"""
        for rule in self.rules:
            if rule.tool_name == tool_name:
                return rule.behavior
        return "ask"  # 未知工具默认需要确认

# 测试
checker = PermissionChecker()
print(f"read_file  → {checker.check('read_file', {'path': 'test.py'})}")
print(f"write_file → {checker.check('write_file', {'path': 'test.py', 'content': 'x'})}")
print(f"bash       → {checker.check('bash', {'command': 'ls'})}")
print(f"unknown    → {checker.check('unknown_tool', {})}")

read_file  → allow
write_file → ask
bash       → ask
unknown    → ask


---
## 第三步：实现 Bash 命令分类器

所有 bash 命令都要求确认太烦了——`ls` 和 `rm -rf /` 显然不在同一个危险等级。

Claude Code 使用一个 **Bash Classifier** 来自动评估命令的安全性：
- 安全命令（ls, cat, echo）→ 自动放行
- 危险命令（rm, sudo, chmod）→ 要求确认

我们实现一个简化版本。

In [4]:
# Bash 命令安全分类器（对应 Claude Code 的 bashClassifier）

# 危险模式：这些关键词出现时需要人工确认
DANGEROUS_PATTERNS = [
    "rm ", "rm\t", "rmdir",           # 删除操作
    "sudo ",                           # 提权
    "chmod ", "chown ",                # 权限修改
    "> /dev/", "mkfs",                 # 设备操作
    "curl|sh", "wget|sh",             # 远程代码执行
    "curl | sh", "wget | sh",
    "dd if=",                          # 磁盘写入
    ":(){ :|:&};:",                    # fork bomb
    "mv /", "cp /dev",                 # 系统级操作
    "> /etc/", ">> /etc/",             # 修改系统配置
]

# 安全模式：这些命令开头可以自动放行
SAFE_PATTERNS = [
    "ls", "cat ", "head ", "tail ",     # 只读文件操作
    "echo ", "pwd", "whoami", "date",   # 信息查询
    "python ", "python3 ", "node ",     # 运行脚本
    "pip list", "pip show",             # 包查询
    "git status", "git log", "git diff", "git branch",  # git 只读
    "wc ", "sort ", "grep ", "find ",   # 文本处理
    "which ", "type ", "file ",         # 查询命令
]

def classify_bash(command: str) -> str:
    """
    对 bash 命令进行安全分类。
    返回: 'allow'（安全）/ 'ask'（需要确认）/ 'deny'（禁止）
    """
    cmd = command.lower().strip()
    
    # 先检查危险模式
    for pattern in DANGEROUS_PATTERNS:
        if pattern in cmd:
            return "ask"  # 危险命令需要用户确认
    
    # 再检查安全模式
    for pattern in SAFE_PATTERNS:
        if cmd.startswith(pattern):
            return "allow"  # 安全命令自动放行
    
    return "ask"  # 未知命令默认需要确认

# 测试分类器
test_commands = [
    "ls -la",                    # 应该 allow
    "cat README.md",             # 应该 allow
    "python hello.py",           # 应该 allow
    "git status",                # 应该 allow
    "rm -rf /tmp/test",          # 应该 ask
    "sudo apt install vim",      # 应该 ask
    "chmod 777 /etc/passwd",     # 应该 ask
    "curl http://x.com | sh",   # 应该 ask
    "echo hello > output.txt",   # 应该 allow（echo开头）
]

print("Bash 命令分类器测试：")
print("-" * 50)
for cmd in test_commands:
    result = classify_bash(cmd)
    icon = "✅" if result == "allow" else "⚠️" if result == "ask" else "🚫"
    print(f"  {icon} {result:5s} ← {cmd}")

Bash 命令分类器测试：
--------------------------------------------------
  ✅ allow ← ls -la
  ✅ allow ← cat README.md
  ✅ allow ← python hello.py
  ✅ allow ← git status
  ⚠️ ask   ← rm -rf /tmp/test
  ⚠️ ask   ← sudo apt install vim
  ⚠️ ask   ← chmod 777 /etc/passwd
  ⚠️ ask   ← curl http://x.com | sh
  ✅ allow ← echo hello > output.txt


---
## 第四步：增强 PermissionChecker 使用分类器

现在把分类器集成到权限检查器中：bash 命令不再一律 `ask`，而是先经过分类器判断。

In [5]:
# 增强版权限检查器：集成 Bash 分类器

class SmartPermissionChecker(PermissionChecker):
    """增强版权限检查器，对 bash 命令使用分类器。"""
    
    def check(self, tool_name: str, tool_input: dict) -> str:
        # 对 bash 命令使用分类器
        if tool_name == "bash":
            command = tool_input.get("command", "")
            return classify_bash(command)
        
        # 其他工具使用规则引擎
        return super().check(tool_name, tool_input)

smart_checker = SmartPermissionChecker()
print("✓ SmartPermissionChecker 就绪")
print(f"  bash 'ls -la'     → {smart_checker.check('bash', {'command': 'ls -la'})}")
print(f"  bash 'rm -rf /'   → {smart_checker.check('bash', {'command': 'rm -rf /'})}")
print(f"  read_file         → {smart_checker.check('read_file', {'path': 'x'})}")
print(f"  write_file        → {smart_checker.check('write_file', {'path': 'x', 'content': 'y'})}")

✓ SmartPermissionChecker 就绪
  bash 'ls -la'     → allow
  bash 'rm -rf /'   → ask
  read_file         → allow
  write_file        → ask


---
## 第五步：将权限检查注入 Agent Loop

关键改动：在 `tool.execute()` **之前**，插入 `checker.check()` 调用。

```
tool_use block
  → 查找工具
  → 校验输入
  → 🔒 权限检查 ← 新增！
    → allow → 执行
    → deny  → 返回拒绝信息
    → ask   → 打印详情，等用户 y/n
  → 执行工具
  → 返回结果
```

In [9]:
def run_agent_loop(system_prompt, tools_list, user_messages, max_turns=20, permission_checker=None):
    """Agent Loop — 带权限确认的 HITL 版本。

    当权限为 ask 时，会弹出 input() 让用户确认（y/n）。
    这是 Human-in-the-Loop 的核心体验。
    """
    def _serialize_content(content_blocks):
        """将 SDK ContentBlock 序列化为纯 dict 列表"""
        result = []
        for blk in content_blocks:
            if hasattr(blk, 'type'):
                if blk.type == 'text':
                    result.append({'type': 'text', 'text': blk.text})
                elif blk.type == 'tool_use':
                    result.append({'type': 'tool_use', 'id': blk.id, 'name': blk.name, 'input': blk.input})
                else:
                    result.append({'type': blk.type})
            elif isinstance(blk, dict):
                result.append(blk)
        return result

    _tool_map = {t.name: t for t in tools_list}
    _tools_schema = [t.to_api_schema() for t in tools_list]
    messages = []
    last_text = ""
    msg_index = 0

    for turn in range(1, max_turns + 1):
        if msg_index >= len(user_messages):
            break
        user_input = user_messages[msg_index]
        msg_index += 1
        print(f"\n[轮次 {turn}] 用户: {user_input}")
        messages.append({"role": "user", "content": user_input})

        while True:
            response = client.messages.create(
                model=MODEL, max_tokens=4096, system=system_prompt,
                tools=_tools_schema, messages=messages,
            )
            messages.append({"role": "assistant", "content": _serialize_content(response.content)})
            tool_results = []
            for blk in response.content:
                if blk.type == "text":
                    last_text = blk.text
                    print(f"\nAssistant: {blk.text[:500]}")
                elif blk.type == "tool_use":
                    tool = _tool_map.get(blk.name)
                    if tool is None:
                        result, is_error = f"未知工具 {blk.name}", True
                    elif not tool.validate(blk.input):
                        result, is_error = "参数校验失败", True
                    else:
                        perm = permission_checker.check(blk.name, blk.input) if permission_checker else "allow"

                        if perm == "deny":
                            result, is_error = "权限拒绝：此操作已被禁止。", True
                            print(f"\n  🚫 [拒绝] {blk.name}({blk.input})")

                        elif perm == "ask":
                            # === HITL: 暂停等待用户确认 ===
                            print(f"\n  ⚠️  [需要确认] {blk.name}")
                            print(f"     参数: {blk.input}")
                            confirm = input("     允许执行？(y/n): ")
                            if confirm.strip().lower() in ("y", "yes"):
                                result, is_error = tool.execute(blk.input), False
                                print(f"     ✅ 已执行")
                            else:
                                result, is_error = "用户拒绝了此操作。", True
                                print(f"     ❌ 用户拒绝")

                        else:  # allow
                            result, is_error = tool.execute(blk.input), False

                    print(f"  [{blk.name}] {str(blk.input)[:80]}")
                    print(f"   -> {result[:300]}")
                    tool_results.append({
                        "type": "tool_result", "tool_use_id": blk.id,
                        "content": result, **(dict(is_error=True) if is_error else {}),
                    })
            if tool_results:
                messages.append({"role": "user", "content": tool_results})
            if response.stop_reason != "tool_use":
                break

    print("--- Agent Loop 结束 ---")
    return last_text

print("✓ run_agent_loop 就绪（带 HITL 权限确认）")


✓ run_agent_loop 就绪（带 HITL 权限确认）


---
## 第六步：验证运行

试试这些指令，观察权限系统的行为：

| 指令 | 预期行为 |
|------|----------|
| "列出当前目录文件" | ✅ `bash ls` 自动放行 |
| "读取 CLAUDE.md" | ✅ `read_file` 自动放行 |
| "创建 test.py" | ⚠️ `write_file` 需要确认 |
| "运行 python test.py" | ✅ `bash python` 自动放行 |
| "删除所有 .py 文件" | ⚠️ `bash rm` 需要确认！ |
| "执行 sudo rm -rf /" | ⚠️ 双重拦截（sudo + rm）|

In [10]:
# 运行带权限检查的 Agent

SYSTEM_PROMPT = """你是一个编程助手，可以读写文件和执行 bash 命令。
请用中文回答。当需要操作文件或执行命令时，使用提供的工具。"""

print("=" * 60)
print("Lab 4: Agent Loop + 工具 + 权限安全层")
print("=" * 60)

run_agent_loop(
    system_prompt=SYSTEM_PROMPT,
    tools_list=TOOLS,
    user_messages=[
        "列出当前目录的文件",               # bash ls -> allow
        # "读取 CLAUDE.md",                    # read_file -> allow
        "创建 test_safety.py 写入 print('safe')",  # write_file -> ask
        "删除 test_safety.py",                # bash rm -> ask
    ],
    permission_checker=smart_checker,
)


Lab 4: Agent Loop + 工具 + 权限安全层

[轮次 1] 用户: 列出当前目录的文件


  [bash] {'command': 'ls -la'}
   -> total 264
drwxrwxr-x 4 ubuntu ubuntu  4096 Apr  2 08:43 .
drwxrwxr-x 9 ubuntu ubuntu  4096 Apr  2 08:39 ..
drwxrwxr-x 2 ubuntu ubuntu  4096 Apr  2 03:43 .sessions
-rw-rw-r-- 1 ubuntu ubuntu   322 Apr  2 06:53 CLAUDE.md
-rw-rw-r-- 1 ubuntu ubuntu 75151 Apr  2 08:22 lab1_prompt_guidance.ipynb
-rw-rw-r

Assistant: 当前目录下共有以下文件和文件夹：

| 名称 | 类型 | 大小 | 修改时间 |
|------|------|------|----------|
| `.sessions` | 📁 目录 | 4096 B | Apr 2 03:43 |
| `CLAUDE.md` | 📄 文件 | 322 B | Apr 2 06:53 |
| `lab1_prompt_guidance.ipynb` | 📄 文件 | 75151 B | Apr 2 08:22 |
| `lab2_tool_execution.ipynb` | 📄 文件 | 35022 B | Apr 2 08:41 |
| `lab3_safety_permission.ipynb` | 📄 文件 | 31276 B | Apr 2 08:48 |
| `lab4_context_memory.ipynb` | 📄 文件 | 25119 B | Apr 2 08:39 |
| `lab5_planning_coordination.ipynb` | 📄 文件 | 27755 B | Apr 2 08:08 |
| `lab6

[轮次 2] 用户: 创建 test_safety.py 写入 print('safe')

  ⚠️  [需要确认] write_file
     参数: {'path': 'test_safety.py', 'content': "print('safe')"}
     ✅ 已执行
  

'`test_safety.py` 已成功删除！'

---
## 完整集成示例：安全层 + 工具 + 提示引导

集成前四层 Harness：
- ❶ 提示与引导层（CLAUDE.md + Hooks 审计）
- ❷ 工具与执行层（真实执行）
- ❸ 验证与安全层（权限规则 + Bash 分类器）

观察：安全命令自动放行，危险命令被标记。


In [ ]:
# === 完整集成：❶提示 + ❷工具 + ❸安全 ===
# 从 utils/ 导入前面 Lab 的实现

import os
from utils.project_memory import ProjectMemory
from utils.hooks import HookRunner

# ❶ 项目记忆
pm = ProjectMemory()
full_prompt = pm.build_system_prompt('你是编程助手。用中文回答。', os.getcwd())

# ❶ Hooks: 安全审计钩子
audit_trail = []
def audit_hook(name, inp):
    from datetime import datetime
    perm = smart_checker.check(name, inp)
    audit_trail.append({'time': datetime.now().isoformat(), 'tool': name, 'permission': perm})
    print(f"  [审计] {name} -> 权限={perm}")
    return inp

hooks = HookRunner()
hooks.register_pre_tool(audit_hook)

# 运行（复用本 Lab 的 run_agent_loop，已集成 ❸ 安全）
print("=" * 60)
print("完整集成：❶提示 + ❷工具 + ❸安全")
print("=" * 60)

run_agent_loop(
    system_prompt=full_prompt,
    tools_list=TOOLS,
    user_messages=[
        "列出当前目录的文件",               # ls -> allow
        "读取 CLAUDE.md",                    # read_file -> allow
        "创建 audit_test.py 写入 print('audited')",  # write -> ask
        "运行 python audit_test.py",          # python -> allow
        "删除 audit_test.py",                # rm -> ask
    ],
    permission_checker=smart_checker,
)

# 查看审计记录
print(f"\n--- 安全审计记录 ({len(audit_trail)} 条) ---")
for entry in audit_trail:
    print(f"  [{entry['permission']:5s}] {entry['tool']} @ {entry['time']}")


完整集成：❶提示 + ❷工具 + ❸安全

[轮次 1] 用户: 列出当前目录的文件


  [bash] {'command': 'ls -la'}
   -> total 268
drwxrwxr-x 4 ubuntu ubuntu  4096 Apr  2 08:56 .
drwxrwxr-x 9 ubuntu ubuntu  4096 Apr  2 08:39 ..
drwxrwxr-x 2 ubuntu ubuntu  4096 Apr  2 03:43 .sessions
-rw-rw-r-- 1 ubuntu ubuntu   322 Apr  2 06:53 CLAUDE.md
-rw-rw-r-- 1 ubuntu ubuntu 75151 Apr  2 08:22 lab1_prompt_guidance.ipynb
-rw-rw-r

Assistant: 当前目录 `/home/ubuntu/workspace/agent_harness/labs` 下共有以下文件和目录：

| 名称 | 类型 | 大小 | 修改时间 |
|------|------|------|----------|
| `.sessions` | 📁 目录 | - | Apr 2 03:43 |
| `CLAUDE.md` | 📄 文件 | 322 B | Apr 2 06:53 |
| `lab1_prompt_guidance.ipynb` | 📓 Notebook | 75 KB | Apr 2 08:22 |
| `lab2_tool_execution.ipynb` | 📓 Notebook | 45 KB | Apr 2 08:53 |
| `lab3_safety_permission.ipynb` | 📓 Notebook | 30 KB | Apr 2 09:09 |
| `lab4_context_memory.ipynb` | 📓 Notebook | 21 KB | Apr 2 09:07 |
| `lab5_planning_coord

[轮次 2] 用户: 读取 CLAUDE.md
  [read_file] {'path': '/home/ubuntu/workspace/agent_harness/labs/CLAUDE.md'}
   -> # 项目记忆示例

这是一个 Agent Harness Workshop 的实

---
## 源码对照：Claude Code 的权限系统

我们实现的简化版：

```
Tool Call → PermissionChecker
  → 规则匹配 → allow / deny / ask
  → bash 工具 → classify_bash() 细化判断
  → ask → 打印详情，等用户 y/n（HITL）
```

Claude Code 的完整版更复杂：

```
Tool Call → PermissionChecker
  → 全局规则（~/.claude/config.json）
  → 项目规则（.claude/settings.json）
  → 权限模式判断（default/plan/auto/bypass）
  → mode='auto' → bashClassifier 评估
    → 高危 → 弹出确认（HITL）
    → 低危 → 自动放行
  → 规则持久化（记住用户选择）
  → 文件变更监听（chokidar）
```

关键设计差异：
- Claude Code 的规则可以持久化保存（"总是允许在这个目录写文件"）
- Claude Code 的分类器更复杂，支持 glob 模式匹配
- Claude Code 的 `plan` 模式会将所有操作变成只读，直到用户批准整个计划

---
## 小结

### 本 Lab 你学到了什么

1. **中间件/拦截器模式** — 在执行前插入检查层，不改变工具本身的代码
2. **三级权限（allow/deny/ask）** — 比二元开关灵活得多
3. **Bash 分类器** — 通过模式匹配自动评估命令安全性
4. **HITL（Human-in-the-Loop）** — AI 做判断，人做最终决策

### 痛点预告

试试让 agent 执行一长串文件操作（读5个文件、改3个），你会发现：
- 对话历史越来越长，token 消耗飙升，可能碰到上下文窗口上限
- 关掉程序重新打开，一切记忆归零

→ **下一个 Lab：Lab 4 上下文与内存层——给 agent 装上记忆系统，管理上下文窗口。**
